In [2]:
import yfinance as yf
import pandas as pd
tickers=pd.read_csv('../../data_collection/data/stock_index/nasdaq100.csv',index_col=0)
tickers = tickers.index.tolist()
data = yf.download(tickers, period="5y")['Close']

[**********************95%*********************  ]  96 of 101 completedHTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ANSS"}}}
[**********************98%********************** ]  99 of 101 completed$ANSS: possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")
[*********************100%***********************]  101 of 101 completed

1 Failed download:
['ANSS']: possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")


In [3]:
#drop na by columns
data = data.dropna(axis=1)
data.head()

Ticker,AAPL,ABNB,ADBE,ADI,ADP,ADSK,AEP,AMAT,AMD,AMGN,...,TSLA,TTD,TTWO,TXN,VRSK,VRTX,WBD,WDAY,XEL,ZS
Date,,,,,,,,,,,,,,,,,,,,,
2021-01-25,139.125793,177.529999,473.440002,140.864822,147.544052,293.640015,68.772461,103.540604,94.129997,219.648285,...,293.600006,78.558998,203.990005,150.017670,182.744644,241.309998,39.000000,234.990005,56.742905,207.660004
2021-01-26,139.359451,192.740005,476.279999,141.982773,148.310272,291.750000,68.131294,101.060303,94.709999,220.929810,...,294.363342,77.628998,199.779999,148.759705,182.570343,237.639999,40.720001,235.600006,56.154140,197.000000
2021-01-27,138.288605,201.250000,460.000000,135.146652,149.337860,273.470001,66.982178,94.404663,88.839996,214.488129,...,288.053345,75.629997,195.380005,141.350784,178.473373,225.250000,43.869999,223.449997,54.456123,199.220001
2021-01-28,133.450577,187.369995,465.670013,136.731979,150.311371,284.220001,67.082077,96.865807,87.519997,211.660324,...,278.476654,79.452003,200.300003,146.816360,180.565430,230.039993,41.020000,228.559998,54.302525,206.279999
2021-01-29,128.456787,183.630005,458.769989,135.009201,148.842072,277.429993,67.373528,92.585144,85.639999,206.260910,...,264.510010,76.598999,200.449997,144.616898,177.727631,229.080002,41.419998,227.529999,54.601177,199.699997


In [ ]:
returns = data.pct_change().dropna()

volatility = returns.std()
avg_return = returns.mean()


from sklearn.cluster import KMeans
import pandas as pd

features = pd.DataFrame({
    'volatility': volatility,
    'avg_return': avg_return
})

kmeans = KMeans(n_clusters=3)
features['cluster'] = kmeans.fit_predict(features)

In [14]:
features

,volatility,avg_return,cluster
Ticker,,,
AAPL,0.017433,0.000612,0
ABNB,0.028685,0.000175,2
ADBE,0.022213,-0.000110,2
ADI,0.020569,0.000827,2
ADP,0.013289,0.000534,0
...,...,...,...
VRTX,0.017929,0.000692,0
WBD,0.035493,0.000384,1
WDAY,0.023265,0.000097,2


In [7]:
#elbow method to find optimal number of clusters
import plotly.express as px
from sklearn.cluster import KMeans
inertia = []
n_samples = features.shape[0]
for i in range(1, n_samples + 1):
    kmeans = KMeans(n_clusters=i)
    kmeans.fit(features[['volatility', 'avg_return']])
    inertia.append(kmeans.inertia_)

fig = px.line(x=range(1, n_samples + 1), y=inertia, labels={'x': 'Number of Clusters', 'y': 'Inertia'})
fig.update_layout(title='Elbow Method for Optimal Clusters')
fig.show()

In [8]:
import plotly.express as px
#also show the ticker in the plot
features['Ticker'] = features.index
# Create a scatter plot with Plotly Express
fig = px.scatter(features, x='volatility', y='avg_return', color='cluster', text='Ticker',
                 title='Volatility vs Average Return Clustering',
                 labels={'volatility': 'Volatility', 'avg_return': 'Average Return'})
# Update layout for better readability
fig.update_traces(textposition='top center')
fig.update_layout(
    xaxis_title='Volatility',
    yaxis_title='Average Return',
    legend_title='Cluster'
)
fig.show()

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled_data = scaler.fit_transform(returns.T)  # T เพื่อให้หุ้นเป็น observations

pca = PCA(n_components=2)
pca_result = pca.fit_transform(scaled_data)

print("Explained variance ratio:", pca.explained_variance_ratio_)

Explained variance ratio: [0.14624463 0.06965678]


In [ ]:
pca_df = pd.DataFrame(pca_result, columns=['PC1', 'PC2'], index=returns.columns)

fig = px.scatter(pca_df, x='PC1', y='PC2', text=pca_df.index,
                 title='PCA of Stock Returns',
                 labels={'PC1': 'Principal Component 1', 'PC2': 'Principal Component 2'})
fig.update_traces(textposition='top center')
fig.update_layout(
    xaxis_title='Principal Component 1',
    yaxis_title='Principal Component 2'
)
fig.show()
